# QLoRA throughput pilot for Qwen3-4B (Milestone 3)

Implements the throughput pilot of `docs/milestone-2/EVAL_PROTOCOL.md`, Amendment 1 ("Epoch budget"): at most one GPU hour on Kaggle, on **train data only**. It measures tokens per second, peak memory and fp16 stability of the declared QLoRA training step on one T4 and on two T4s (data parallel), fixes the training library versions, and applies the declared budget rule. The pilot's weights are never saved or evaluated.

Steps:

1. unpack the bundle (a commit plus manifest-checked datasets);
2. build the training data from the train split and check it against the committed manifest (`docs/milestone-3/train-v1.manifest.json`);
3. count tokens per epoch with the model's own chat template;
4. time the training step on one T4, then on two (12 minutes each). The first optimiser step uses the 16 longest examples, for peak memory, and is not timed, nor is the second; the timed steps take a seeded random sample of the training file (seed 2029);
5. apply the budget rule (`budget.json`), which reports the timed sample's mean tokens per example next to the dataset's.

**Setup:** Accelerator *GPU T4 x2*, Internet *on*, attach the private Kaggle Dataset with the Milestone 3 bundle (`hetren-m3-bundle.tar.gz`), then *Save Version → Save & Run All*. Download `qlora-pilot.tar.gz` from the Output tab; the pilot and its budget are logged in `docs/milestone-3/VALIDATION_LOG.md`. A pilot that fails (out of memory, a library error, a timeout) is a result too: it is recorded and the notebook goes on.

In [ ]:
import glob, json, os, shutil, subprocess, sys, tarfile

WORK = "/kaggle/working/hetren"
OUT = "/kaggle/working/qlora-pilot"
DATA = "/kaggle/working/train-v1"
MINUTES = 12

if not os.path.exists(WORK):
    unpacked = glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True)
    packed = glob.glob("/kaggle/input/**/hetren-*bundle.tar.gz", recursive=True)

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)
os.chdir(WORK)
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Pinned versions: the repo's runtime requirements, transformers and accelerate as in evaluation,
# and the QLoRA libraries this pilot fixes (Amendment 1).
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1 bitsandbytes==0.50.2
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, bitsandbytes; n = torch.cuda.device_count(); print(n, 'GPUs:', [torch.cuda.get_device_name(i) for i in range(n)]); assert n == 2"

## Training data (train split only), checked against the committed manifest

In [ ]:
subprocess.run([sys.executable, "-m", "packages.support_evaluator.training_data", "--out", DATA,
                "--check", "docs/milestone-3/train-v1.manifest.json"], check=True)
subprocess.run([sys.executable, "-m", "packages.hetren_baselines.qlora", "tokens",
                "--data", f"{DATA}/train.jsonl", "--out", OUT], check=True)

## Throughput: one T4, then two T4s (data parallel)

In [ ]:
def run_pilot(configuration, command, visible):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": visible}
    try:
        code = subprocess.run(command, env=env, timeout=(MINUTES + 20) * 60).returncode
    except subprocess.TimeoutExpired:
        code = "timeout"

    path = f"{OUT}/pilot-{configuration}.json"

    if not os.path.exists(path):  # the process died before it could record its result
        json.dump({"configuration": configuration, "completed": False, "non_finite_losses": 0,
                   "tokens_per_second": 0.0, "error": f"exit {code}"}, open(path, "w"), indent=2)

    print(configuration, "exit", code)


pilot = ["-m", "packages.hetren_baselines.qlora", "pilot", "--data", f"{DATA}/train.jsonl", "--out", OUT,
         "--minutes", str(MINUTES)]
run_pilot("1xGPU", [sys.executable, *pilot], "0")
run_pilot("2xGPU", ["torchrun", "--nproc_per_node", "2", *pilot], "0,1")

## Budget rule and outputs

In [ ]:
subprocess.run([sys.executable, "-m", "packages.hetren_baselines.qlora", "budget", "--out", OUT], check=True)
json.dump({"bundle_commit": COMMIT}, open(f"{OUT}/bundle.json", "w"))

with tarfile.open("/kaggle/working/qlora-pilot.tar.gz", "w:gz") as results:
    results.add(OUT, arcname=os.path.basename(OUT))

for name in sorted(os.listdir(OUT)):
    print("==", name)
    print(open(f"{OUT}/{name}").read())